Regression Tree:

SSE_split = sum_i in L(y_i - bar y_L)^2 + sum_ i in R(y_i - bar y_R)^2

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

X_demo = np.array([
    [1.0], [2.0], [3.0],
    [7.0], [8.0], [9.0]
])
y_demo = np.array([
    2.0, 3.0, 2.5,
    12.0, 13.0, 12.5
])

def calculate_sse(y):
    y = np.asarray(y, dtype=float)

    if len(y) == 0:
        return 0.0

    return float(np.sum((y - y.mean()) ** 2))

left_mask = X_demo[:, 0] <= 5.0

y_left = y_demo[left_mask]
y_right = y_demo[~left_mask]

parent_sse = calculate_sse(y_demo)
split_sse = calculate_sse(y_left) + calculate_sse(y_right)
print("Parent SSE:", round(parent_sse, 4))
print("Left values:", y_left)
print("Right values:", y_right)
print("Split SSE:", round(split_sse, 4))
print("SSE reduction:", round(parent_sse - split_sse, 4))

assert split_sse < parent_sse
print("Regression split test: PASSED")



Parent SSE: 151.0
Left values: [2.  3.  2.5]
Right values: [12.  13.  12.5]
Split SSE: 1.0
SSE reduction: 150.0
Regression split test: PASSED


In [19]:
def find_best_split(X, y, min_samples_leaf=1, max_features=None, rng=None):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n_samples, n_features = X.shape

    if rng is None:
        rng = np.random.default_rng(42)

    if max_features is None:
        max_features = n_features

    feature_indices = rng.choice(n_features, size=max_features, replace=False)

    best_split = None
    best_sse = float('inf')

    for feature_idx in feature_indices:
        unique_values = np.unique(X[:, feature_idx])

        if len(unique_values) < 2:
            continue
        
        thresholds = (unique_values[:-1] + unique_values[1:]) / 2

        for threshold in thresholds:
            left_mask = X[:, feature_idx] <= threshold
            y_left = y[left_mask]
            y_right = y[~left_mask]

            if (len(y_left) < min_samples_leaf or len(y_right) < min_samples_leaf):
                continue

            split_sse = (calculate_sse(y_left) + calculate_sse(y_right))

            if split_sse < best_sse:
                best_sse = split_sse

                best_split = {
                    'feature_idx': feature_idx,
                    'threshold': float(threshold),
                    'sse': float(split_sse),
                    'left_count': len(y_left),
                    'right_count': len(y_right)
                }

    return best_split

In [20]:
class TreeNode:

    def __init__(self, feature_idx=None, threshold=None, left=None, right=None, value=None):
        self.feature_idx = feature_idx
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf(self):
        return self.value is not None
    
def bootstrap_sample(X,y, rng):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    n_samples = len(y)
    indices = rng.choice(n_samples, size=n_samples, replace=True)

    return X[indices], y[indices], indices

        

In [21]:
class DecisionTreeRegressor:
    """Regression decision tree"""

    def __init__(self, max_depth=3, min_samples_split=2, min_samples_leaf=1, max_features=None, random_state=42):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state
        self.root = None
        self.rng = None

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        if X.ndim != 2 or y.ndim != 1:
            raise ValueError("Expected X 2D and y 1D")

        if len(X) != len(y) or len(y) == 0:
            raise ValueError("invalid shapes")

        self.rng = np.random.default_rng(self.random_state)
        self.root = self._grow_tree(X,y, depth=0)
        return self

    def _grow_tree(self, X, y, depth):
        y_pred = float(np.mean(y))

        if(depth >= self.max_depth or len(y) < self.min_samples_split or calculate_sse(y) <= 1e-12):
            return TreeNode(value=y_pred)

        split = find_best_split(X,y, min_samples_leaf=self.min_samples_leaf, max_features=self.max_features, rng=self.rng)
        if split is None:
            return TreeNode(value=y_pred)

        parent_sse = calculate_sse(y)
        if parent_sse - split['sse'] <= 1e-12:
            return TreeNode(value=y_pred)

        feature_idx = split['feature_idx']
        threshold = split['threshold']

        left_mask = X[:, feature_idx] <= threshold
        left_child = self._grow_tree(X[left_mask], y[left_mask], depth + 1)
        right_child = self._grow_tree(X[~left_mask], y[~left_mask], depth + 1)

        return TreeNode(feature_idx=feature_idx, threshold=threshold, left=left_child, right=right_child)

    def _predict_one(self, x, node):
        """traverse the tree"""
        while not node.is_leaf():
            if x[node.feature_idx] <= node.threshold:
                node = node.left
            else:
                node = node.right

        return float(node.value)

    def predict(self, X):
        if self.root is None:
            raise ValueError("Tree must be fitted")

        X = np.asarray(X, dtype=float)

        return np.array([self._predict_one(x, self.root) for x in X])



In [22]:
class RandomForest:
    
    def __init__(self, n_estimators=10, max_depth=5, min_samples_split=2, min_samples_leaf=1, max_features=None, random_state=42):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.min_samples_leaf = min_samples_leaf
        self.max_features = max_features
        self.random_state = random_state
        self.trees = []

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float)

        rng = np.random.default_rng(self.random_state)
        self.trees = []

        for _ in range(self.n_estimators):
            X_boot, y_boot, _ = bootstrap_sample(X, y, rng)
            tree_seed = int(rng.integers(0, np.iinfo(np.int32).max))
            tree = DecisionTreeRegressor(max_depth=self.max_depth, min_samples_split=self.min_samples_split, min_samples_leaf=self.min_samples_leaf, max_features= self.max_features, random_state= tree_seed)

            tree.fit(X_boot, y_boot)
            self.trees.append(tree)

        return self

    def predict(self, X):
        X = np.asarray(X, dtype = float)

        tree_pred = np.array([tree.predict(X) for tree in self.trees])
        return tree_pred.mean(axis=0)
    


In [23]:

X_random_test = np.column_stack([
    np.arange(10),
    np.arange(10) * 2,
    np.arange(10) ** 2,
    np.arange(10) + 5,
    np.arange(10) * 3
]).astype(float)

y_random_test = np.array([
    1, 1, 2, 2, 3, 3, 8, 8, 9, 9
], dtype=float)

seed = 42

# Reproduce which features should be sampled
expected_rng = np.random.default_rng(seed)
expected_features = expected_rng.choice(
    X_random_test.shape[1],
    size=2,
    replace=False
)

split = find_best_split(
    X_random_test,
    y_random_test,
    min_samples_leaf=2,
    max_features=2,
    rng=np.random.default_rng(seed)
)

print("Sampled feature indices:", expected_features)
print("Selected split feature:", split["feature_idx"])
print("Selected threshold:", split["threshold"])
print("Left samples:", split["left_count"])
print("Right samples:", split["right_count"])

assert split["feature_idx"] in expected_features
assert split["left_count"] >= 2
assert split["right_count"] >= 2

print("Random feature selection: PASSED")

forest = RandomForest(
    n_estimators=5,
    max_depth=3,
    min_samples_leaf=2,
    max_features=2,
    random_state=42
)

forest.fit(X_random_test, y_random_test)

predictions = forest.predict(X_random_test)

print("Number of trees:", len(forest.trees))
print("Predictions:", np.round(predictions, 3))
print("Prediction shape:", predictions.shape)

assert len(forest.trees) == 5
assert predictions.shape == y_random_test.shape
assert np.isfinite(predictions).all()

print("Random Forest initial test: PASSED")


Sampled feature indices: [0 3]
Selected split feature: 0
Selected threshold: 5.5
Left samples: 6
Right samples: 4
Random feature selection: PASSED
Number of trees: 5
Predictions: [1.367 1.367 2.    2.    2.633 2.833 7.    8.1   8.9   8.9  ]
Prediction shape: (10,)
Random Forest initial test: PASSED


In [24]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

FIGURE_DIR = PROJECT_ROOT / "results" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

def save_figure(fig, filename):
    fig.savefig(
        FIGURE_DIR / f"{filename}.png",
        dpi=300,
        bbox_inches="tight"
    )
    print(f"Saved: {filename}")

DATA_PATH = (PROJECT_ROOT / 'data' / 'processed' / 'monthly_features_seasonal.csv')
df = pd.read_csv(DATA_PATH, dtype={'STATION':str})

features_A = [
    "TEMP_MEAN",
    "DEWP_MEAN",
    "WDSP_MEAN",
    "TEMP_MIN_MEAN",
    "TEMP_MAX_MEAN",
    "MONTH_COS",
    "MONTH_SIN"
]

features_B = features_A + [
    "PRCP_LAG_0",
    "PRCP_LAG_1",
    "PRCP_LAG_2"
]

target = 'TARGET_NEXT_MONTH'

print("Dataset:", df.shape)
print("Stations:", df["STATION"].nunique())
print("Feature Set A:", len(features_A))
print("Feature Set B:", len(features_B))

Dataset: (1768, 45)
Stations: 6
Feature Set A: 7
Feature Set B: 10


In [25]:
stations = sorted(df['STATION'].unique())
test_station = stations[0]

train_df = df[df['STATION'] != test_station].copy()
test_df = df[df["STATION"] == test_station].copy()


X_train = train_df[features_A].to_numpy(dtype=float)
y_train = train_df[target].to_numpy(dtype=float)

print("Held-out station:", test_station)
print("Training observations:", len(train_df))
print("Test observations:", len(test_df))
print("Training features:", X_train.shape[1])

Held-out station: 41184099999
Training observations: 1471
Test observations: 297
Training features: 7


In [30]:
import time
from src.models.random_forest import RandomForest as RF

benchmark_model = RF(
    n_estimators=1,
    max_depth=3,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features=3,
    random_state=42
)

start_time = time.perf_counter()

benchmark_model.fit(X_train, y_train)

elapsed_time = time.perf_counter() - start_time

train_predictions = benchmark_model.predict(X_train)

print(f"Training time: {elapsed_time:.3f} seconds")
print("Number of trees:", len(benchmark_model.trees))
print("Prediction shape:", train_predictions.shape)
print("Finite predictions:", np.isfinite(train_predictions).all())

assert len(benchmark_model.trees) == 1
assert train_predictions.shape == y_train.shape
assert np.isfinite(train_predictions).all()

print("Random Forest benchmark: PASSED")


Training time: 0.214 seconds
Number of trees: 1
Prediction shape: (1471,)
Finite predictions: True
Random Forest benchmark: PASSED


In [29]:
from src.validation import manual_loso

outer_folds = list(manual_loso(df))


for fold_number, (test_station, train_df, test_df) in enumerate(
    outer_folds, start=1
):
    print(
        f"Fold {fold_number} | "
        f"Test station: {test_station} | "
        f"Train: {len(train_df)} | "
        f"Test: {len(test_df)}"
    )

outer_station, outer_train, outer_test = outer_folds[0]

inner_folds = list(manual_loso(outer_train))

print("Outer test station", outer_station)
print("Outer training observations:", len(outer_train))
print("Outer testing observations:", len(outer_test))
print("Number of inner folds:", len(inner_folds))

for fold_number, (val_station, inner_train, inner_val) in enumerate(inner_folds, start= 1):
    train_stations = set(inner_train['STATION'])
    val_stations = set(inner_val['STATION'])
    test_stations = set(outer_test['STATION'])

    print(
        f"Inner Fold {fold_number} | "
        f"Validation: {val_station} | "
        f"Train: {len(inner_train)} | "
        f"Validation: {len(inner_val)}"
    )


print("Nested LOSO using shared validation: PASSED")

Fold 1 | Test station: 41184099999 | Train: 1471 | Test: 297
Fold 2 | Test station: 41194099999 | Train: 1455 | Test: 313
Fold 3 | Test station: 41196099999 | Train: 1473 | Test: 295
Fold 4 | Test station: 41198099999 | Train: 1496 | Test: 272
Fold 5 | Test station: 41217099999 | Train: 1466 | Test: 302
Fold 6 | Test station: 41218099999 | Train: 1479 | Test: 289
Outer test station 41184099999
Outer training observations: 1471
Outer testing observations: 297
Number of inner folds: 5
Inner Fold 1 | Validation: 41194099999 | Train: 1158 | Validation: 313
Inner Fold 2 | Validation: 41196099999 | Train: 1176 | Validation: 295
Inner Fold 3 | Validation: 41198099999 | Train: 1199 | Validation: 272
Inner Fold 4 | Validation: 41217099999 | Train: 1169 | Validation: 302
Inner Fold 5 | Validation: 41218099999 | Train: 1182 | Validation: 289
Nested LOSO using shared validation: PASSED


In [37]:
def tune_random_forest(outer_train, feature_cols, param_grid):

    inner_folds = list(manual_loso(outer_train))
    tuning_results = []

    for config_id, params in enumerate(param_grid, start=1):
        fold_scores = []

        for val_station, inner_train, inner_val in inner_folds:
            X_train = inner_train[feature_cols].to_numpy(dtype=float)
            y_train = inner_train[target].to_numpy(dtype=float)
            X_val = inner_train[feature_cols].to_numpy(dtype=float)
            y_val = inner_train[target].to_numpy(dtype=float)

            model = RF(**params)

            model.fit(X_train, y_train)
            y_pred = model.predict(X_val)
            mae = np.mean(np.abs(y_val - y_pred))
            rmse = np.sqrt(np.mean((y_val - y_pred) ** 2))

            fold_scores.append({
                'station': val_station,
                'mae': mae,
                'rmse': rmse
            })

        scores_df = pd.DataFrame(fold_scores)

        tuning_results.append({
            'config_id': config_id,
            **params,
            'mean_mae': scores_df['mae'].mean(),
            'mean_rmse': scores_df['rmse'].mean(),
            'n_inner_folds': len(fold_scores)
        })

    tuning_df = (
        pd.DataFrame(tuning_results)
        .sort_values(
            ["mean_mae", "config_id"]
        )
        .reset_index(drop=True)
    )

    best_config_id = int(tuning_df.iloc[0]["config_id"])
    best_params = param_grid[best_config_id - 1].copy()

    return best_params, tuning_df


In [38]:
from itertools import product

rf_grid = []

for depth, leaf, max_feat in product(
    [3,5],
    [5,10],
    [2,3]
):
    rf_grid.append({
        'n_estimators': 15,
        'max_depth': depth,
        'min_samples_split': 10,
        'min_samples_leaf': leaf,
        'max_features': max_feat,
        'random_state': 42
    })

rf_grid_df = pd.DataFrame(rf_grid)
display(rf_grid_df)

assert len(rf_grid) == 8
assert rf_grid_df.duplicated().sum() == 0

print("Hyperparameter grid: PASSED")
print("Configurations:", len(rf_grid))
print("Inner fits per outer fold:", len(rf_grid) * 5)
print("Total fits per feature set:", len(rf_grid) * 5 * 6)

,n_estimators,max_depth,min_samples_split,min_samples_leaf,max_features,random_state
0,15,3,10,5,2,42
1,15,3,10,5,3,42
2,15,3,10,10,2,42
3,15,3,10,10,3,42
4,15,5,10,5,2,42
5,15,5,10,5,3,42
6,15,5,10,10,2,42
7,15,5,10,10,3,42


Hyperparameter grid: PASSED
Configurations: 8
Inner fits per outer fold: 40
Total fits per feature set: 240


In [39]:

outer_station, outer_train, outer_test = outer_folds[0]

best_params_A, tuning_A = tune_random_forest(
    outer_train=outer_train,
    feature_cols=features_A,
    param_grid=rf_grid
)

print("Held-out outer station:", outer_station)
print("Best hyperparameters:", best_params_A)

display(tuning_A)

assert len(tuning_A) == 8
assert (tuning_A["n_inner_folds"] == 5).all()
assert np.isfinite(
    tuning_A[["mean_mae", "mean_rmse"]].to_numpy()
).all()

print("First outer-fold tuning: PASSED")


Held-out outer station: 41184099999
Best hyperparameters: {'n_estimators': 15, 'max_depth': 5, 'min_samples_split': 10, 'min_samples_leaf': 5, 'max_features': 3, 'random_state': 42}


,config_id,n_estimators,max_depth,min_samples_split,min_samples_leaf,max_features,random_state,mean_mae,mean_rmse,n_inner_folds
0,6,15,5,10,5,3,42,7.143159,18.115787,5
1,5,15,5,10,5,2,42,7.189194,18.266231,5
2,8,15,5,10,10,3,42,7.342829,18.982615,5
3,7,15,5,10,10,2,42,7.424588,19.147430,5
4,2,15,3,10,5,3,42,7.834168,19.447775,5
5,1,15,3,10,5,2,42,7.849188,19.621876,5
6,3,15,3,10,10,2,42,7.889106,19.924043,5
7,4,15,3,10,10,3,42,7.897097,19.900615,5


First outer-fold tuning: PASSED


In [ ]:

rf_verified_results_A = []

for fold_number, (test_station, outer_train, outer_test) in enumerate(
    outer_folds, start=1
):

    # Use the hyperparameters selected previously
    station_tuning = (
        rf_tuning_A_df[
            rf_tuning_A_df["TEST_STATION"] == test_station
        ]
        .sort_values(["mean_mae", "config_id"])
    )

    best = station_tuning.iloc[0]

    params = {
        "n_estimators": int(best["n_estimators"]),
        "max_depth": int(best["max_depth"]),
        "min_samples_split": int(best["min_samples_split"]),
        "min_samples_leaf": int(best["min_samples_leaf"]),
        "max_features": int(best["max_features"]),
        "random_state": int(best["random_state"])
    }

    X_train = outer_train[features_A].to_numpy(dtype=float)
    y_train = outer_train[target].to_numpy(dtype=float)

    X_test = outer_test[features_A].to_numpy(dtype=float)
    y_test = outer_test[target].to_numpy(dtype=float)

    model = RF(**params)
    model.fit(X_train, y_train)

    # Use a unique variable for the predictions
    station_predictions = model.predict(X_test)

    mae = np.mean(np.abs(y_test - station_predictions))
    rmse = np.sqrt(
        np.mean((y_test - station_predictions) ** 2)
    )

    ss_res = np.sum((y_test - station_predictions) ** 2)
    ss_tot = np.sum((y_test - y_test.mean()) ** 2)

    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    rf_verified_results_A.append({
        "FOLD": fold_number,
        "TEST_STATION": test_station,
        "FEATURE_SET": "A",
        "TRAIN_ROWS": len(outer_train),
        "TEST_ROWS": len(outer_test),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "N_ESTIMATORS": params["n_estimators"],
        "MAX_DEPTH": params["max_depth"],
        "MIN_SAMPLES_LEAF": params["min_samples_leaf"],
        "MAX_FEATURES": params["max_features"],
        "PRED_MIN": float(station_predictions.min()),
        "PRED_MAX": float(station_predictions.max()),
        "NEGATIVE_PREDICTIONS": int(
            np.sum(station_predictions < 0)
        )
    })

rf_verified_results_A_df = pd.DataFrame(rf_verified_results_A)

display(rf_verified_results_A_df)



,FOLD,TEST_STATION,FEATURE_SET,TRAIN_ROWS,TEST_ROWS,MAE,RMSE,R2,N_ESTIMATORS,MAX_DEPTH,MIN_SAMPLES_LEAF,MAX_FEATURES,PRED_MIN,PRED_MAX,NEGATIVE_PREDICTIONS
0,1,41184099999,A,1471,297,9.995189,18.932939,0.047850,15,5,5,3,0.713783,28.537186,0
1,2,41194099999,A,1455,313,6.581934,13.745526,0.030823,15,5,5,3,0.283159,39.180120,0
2,3,41196099999,A,1473,295,9.242824,20.016753,0.018707,15,5,5,3,0.395186,21.414741,0
3,4,41198099999,A,1496,272,10.251972,35.146901,0.026719,15,5,5,3,0.626573,23.622142,0
4,5,41217099999,A,1466,302,7.072676,13.781070,-0.068525,15,5,5,3,0.814814,36.601967,0
5,6,41218099999,A,1479,289,10.080389,18.096325,-0.120027,15,5,5,3,1.551820,27.121535,0


All outer-fold evaluation metrics: VERIFIED


,TEST_STATION,MAE,RMSE,R2,PRED_MIN,PRED_MAX
0,41184099999,9.995189,18.932939,0.047850,0.713783,28.537186
1,41194099999,6.581934,13.745526,0.030823,0.283159,39.180120
2,41196099999,9.242824,20.016753,0.018707,0.395186,21.414741
3,41198099999,10.251972,35.146901,0.026719,0.626573,23.622142
4,41217099999,7.072676,13.781070,-0.068525,0.814814,36.601967
5,41218099999,10.080389,18.096325,-0.120027,1.551820,27.121535


In [50]:
rf_results_B = []
rf_tuning_B = []

for fold_number, (test_station, outer_train, outer_test) in enumerate(
    outer_folds, start=1
):

    best_params, tuning_df = tune_random_forest(
        outer_train=outer_train,
        feature_cols=features_B,
        param_grid=rf_grid
    )

    tuning_df = tuning_df.copy()
    tuning_df["OUTER_FOLD"] = fold_number
    tuning_df["TEST_STATION"] = test_station
    tuning_df["FEATURE_SET"] = "B"
    rf_tuning_B.append(tuning_df)

    X_train_B = outer_train[features_B].to_numpy(dtype=float)
    y_train_B = outer_train[target].to_numpy(dtype=float)

    X_test_B = outer_test[features_B].to_numpy(dtype=float)
    y_test_B = outer_test[target].to_numpy(dtype=float)

    rf_model_B = RF(**best_params)
    rf_model_B.fit(X_train_B, y_train_B)

    station_pred_B = rf_model_B.predict(X_test_B)

    mae_B = np.mean(np.abs(y_test_B - station_pred_B))
    rmse_B = np.sqrt(
        np.mean((y_test_B - station_pred_B) ** 2)
    )

    ss_res = np.sum((y_test_B - station_pred_B) ** 2)
    ss_tot = np.sum((y_test_B - y_test_B.mean()) ** 2)

    r2_B = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    rf_results_B.append({
        "FOLD": fold_number,
        "TEST_STATION": test_station,
        "FEATURE_SET": "B",
        "TRAIN_ROWS": len(outer_train),
        "TEST_ROWS": len(outer_test),
        "MAE": mae_B,
        "RMSE": rmse_B,
        "R2": r2_B,
        "N_ESTIMATORS": best_params["n_estimators"],
        "MAX_DEPTH": best_params["max_depth"],
        "MIN_SAMPLES_SPLIT": best_params["min_samples_split"],
        "MIN_SAMPLES_LEAF": best_params["min_samples_leaf"],
        "MAX_FEATURES": best_params["max_features"],
        "PRED_MIN": float(station_pred_B.min()),
        "PRED_MAX": float(station_pred_B.max()),
        "NEGATIVE_PREDICTIONS": int(np.sum(station_pred_B < 0))
    })

    print(
        f"Fold {fold_number}/6 | "
        f"Station {test_station} | "
        f"MAE={mae_B:.4f} | "
        f"RMSE={rmse_B:.4f} | "
        f"R2={r2_B:.4f}"
    )

rf_results_B_df = pd.DataFrame(rf_results_B)
rf_tuning_B_df = pd.concat(rf_tuning_B, ignore_index=True)

display(rf_results_B_df)

assert len(rf_results_B_df) == 6
assert rf_results_B_df["TEST_STATION"].nunique() == 6
assert rf_results_B_df["TEST_ROWS"].sum() == len(df)
assert len(rf_tuning_B_df) == 6 * len(rf_grid)

assert np.isfinite(
    rf_results_B_df[["MAE", "RMSE", "R2"]].to_numpy()
).all()

assert (rf_results_B_df["NEGATIVE_PREDICTIONS"] == 0).all()

assert (
    set(rf_results_B_df["TEST_STATION"])
    == set(rf_verified_results_A_df["TEST_STATION"])
)

print("Full Random Forest LOSO — Feature Set B: PASSED")



Fold 1/6 | Station 41184099999 | MAE=9.9676 | RMSE=18.3303 | R2=0.1075
Fold 2/6 | Station 41194099999 | MAE=6.4856 | RMSE=13.6113 | R2=0.0497
Fold 3/6 | Station 41196099999 | MAE=9.5397 | RMSE=20.2208 | R2=-0.0014
Fold 4/6 | Station 41198099999 | MAE=10.9540 | RMSE=34.9553 | R2=0.0373
Fold 5/6 | Station 41217099999 | MAE=6.3779 | RMSE=13.6117 | R2=-0.0424
Fold 6/6 | Station 41218099999 | MAE=9.8277 | RMSE=18.1242 | R2=-0.1235


,FOLD,TEST_STATION,FEATURE_SET,TRAIN_ROWS,TEST_ROWS,MAE,RMSE,R2,N_ESTIMATORS,MAX_DEPTH,MIN_SAMPLES_SPLIT,MIN_SAMPLES_LEAF,MAX_FEATURES,PRED_MIN,PRED_MAX,NEGATIVE_PREDICTIONS
0,1,41184099999,B,1471,297,9.967570,18.330270,0.107502,15,5,10,5,3,0.890015,48.524823,0
1,2,41194099999,B,1455,313,6.485594,13.611299,0.049659,15,5,10,5,3,0.665474,46.484610,0
2,3,41196099999,B,1473,295,9.539658,20.220786,-0.001400,15,5,10,5,3,0.209543,41.858059,0
3,4,41198099999,B,1496,272,10.954028,34.955266,0.037303,15,5,10,5,3,0.611137,25.535500,0
4,5,41217099999,B,1466,302,6.377937,13.611678,-0.042419,15,5,10,5,3,0.891219,35.175619,0
5,6,41218099999,B,1479,289,9.827705,18.124195,-0.123479,15,5,10,5,3,1.164420,34.684693,0


Full Random Forest LOSO — Feature Set B: PASSED


In [51]:

rf_combined_results = pd.concat(
    [rf_verified_results_A_df, rf_results_B_df],
    ignore_index=True
)

rf_feature_summary = (
    rf_combined_results
    .groupby("FEATURE_SET", as_index=False)
    .agg(
        MEAN_MAE=("MAE", "mean"),
        MEAN_RMSE=("RMSE", "mean"),
        MEAN_R2=("R2", "mean")
    )
)

display(rf_feature_summary)


,FEATURE_SET,MEAN_MAE,MEAN_RMSE,MEAN_R2
0,A,8.870831,19.953252,-0.010742
1,B,8.858749,19.808916,0.004528


In [52]:
paired_rf = rf_verified_results_A_df[
    ["TEST_STATION", "TEST_ROWS", "MAE", "RMSE", "R2"]
].merge(
    rf_results_B_df[
        ["TEST_STATION", "TEST_ROWS", "MAE", "RMSE", "R2"]
    ],
    on="TEST_STATION",
    suffixes=("_A", "_B"),
    validate="one_to_one"
)

assert (
    paired_rf["TEST_ROWS_A"] == paired_rf["TEST_ROWS_B"]
).all()

# Negative difference means Feature Set B has lower error
paired_rf["DELTA_MAE_B_MINUS_A"] = (
    paired_rf["MAE_B"] - paired_rf["MAE_A"]
)

paired_rf["DELTA_RMSE_B_MINUS_A"] = (
    paired_rf["RMSE_B"] - paired_rf["RMSE_A"]
)

paired_rf["B_BETTER_MAE"] = (
    paired_rf["DELTA_MAE_B_MINUS_A"] < 0
)

paired_rf["B_BETTER_RMSE"] = (
    paired_rf["DELTA_RMSE_B_MINUS_A"] < 0
)

display(paired_rf.round(4))

print(
    "Stations improved in MAE:",
    paired_rf["B_BETTER_MAE"].sum(), "/ 6"
)

print(
    "Stations improved in RMSE:",
    paired_rf["B_BETTER_RMSE"].sum(), "/ 6"
)


,TEST_STATION,TEST_ROWS_A,MAE_A,RMSE_A,R2_A,TEST_ROWS_B,MAE_B,RMSE_B,R2_B,DELTA_MAE_B_MINUS_A,DELTA_RMSE_B_MINUS_A,B_BETTER_MAE,B_BETTER_RMSE
0,41184099999,297,9.9952,18.9329,0.0478,297,9.9676,18.3303,0.1075,-0.0276,-0.6027,True,True
1,41194099999,313,6.5819,13.7455,0.0308,313,6.4856,13.6113,0.0497,-0.0963,-0.1342,True,True
2,41196099999,295,9.2428,20.0168,0.0187,295,9.5397,20.2208,-0.0014,0.2968,0.2040,False,False
3,41198099999,272,10.2520,35.1469,0.0267,272,10.9540,34.9553,0.0373,0.7021,-0.1916,False,True
4,41217099999,302,7.0727,13.7811,-0.0685,302,6.3779,13.6117,-0.0424,-0.6947,-0.1694,True,True
5,41218099999,289,10.0804,18.0963,-0.1200,289,9.8277,18.1242,-0.1235,-0.2527,0.0279,True,False


Stations improved in MAE: 4 / 6
Stations improved in RMSE: 4 / 6


In [53]:

RESULTS_DIR = PROJECT_ROOT / "results" / "tables"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

rf_combined_results.to_csv(
    RESULTS_DIR / "rf_nested_loso_results.csv",
    index=False
)

pd.concat(
    [rf_tuning_A_df, rf_tuning_B_df],
    ignore_index=True
).to_csv(
    RESULTS_DIR / "rf_nested_loso_tuning.csv",
    index=False
)

rf_feature_summary.to_csv(
    RESULTS_DIR / "rf_feature_summary.csv",
    index=False
)

paired_rf.to_csv(
    RESULTS_DIR / "rf_feature_paired_comparison.csv",
    index=False
)

print("Random Forest research tables exported: PASSED")


Random Forest research tables exported: PASSED


In [ ]:

# Combine Ridge and baseline summaries
ridge_summary = ridge_comparison.rename(
    columns={
        "Model": "MODEL",
        "MAE (mm)": "MAE",
        "RMSE (mm)": "RMSE"
    }
)[["MODEL", "MAE", "RMSE"]].copy()

# Combine Random Forest summaries
rf_summary = rf_feature_summary.rename(
    columns={
        "MEAN_MAE": "MAE",
        "MEAN_RMSE": "RMSE"
    }
)[["FEATURE_SET", "MAE", "RMSE"]].copy()

rf_summary["MODEL"] = (
    "Random Forest - Feature Set "
    + rf_summary["FEATURE_SET"]
)

rf_summary = rf_summary[["MODEL", "MAE", "RMSE"]]

# Final comparison table
model_comparison = pd.concat(
    [ridge_summary, rf_summary],
    ignore_index=True
).sort_values("RMSE").reset_index(drop=True)

display(model_comparison.round(4))

assert len(model_comparison) == 6
assert model_comparison["MODEL"].is_unique
assert model_comparison[["MAE", "RMSE"]].notna().all().all()

model_comparison.to_csv(
    RESULTS_DIR / "ridge_rf_baseline_comparison.csv",
    index=False
)

print("Ridge + Random Forest comparison: PASSED")


,MODEL,MAE,RMSE
0,Calendar-Month Climatology,8.5853,19.6427
1,Random Forest - Feature Set B,8.8587,19.8089
2,Ridge - Feature Set B,8.8251,19.8165
3,Ridge - Feature Set A,8.8414,19.8181
4,Random Forest - Feature Set A,8.8708,19.9533
5,Zero Baseline,6.0550,20.8769


Overall model comparison: PASSED
